# Packing and a Small Mutation Scan

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/uw-ipd/tmol/blob/kdidi/sphinx-docs/docs/tutorial/04_packing_and_mutation_scan.ipynb)

This tutorial first performs fixed-sequence side-chain repacking on a ten-residue ubiquitin slice. It then demonstrates a deliberately small mutation scan using a clearly marked helper built from inspected `PackerTask` mask fields—TMol does not provide a built-in point-mutation-scan protocol.

> **Prerequisite.** Read [Scoring and Analysis](03_scoring_and_analysis.ipynb) first. The target-interaction value below uses its two-orientation block-pair accounting and remains a serial prototype, not a GPU-batched mutation protocol.

## Goals

- Configure `PackerPalette`, `PackerTask`, and the supported conformer samplers.
- Restrict repacking to a selected region while preserving sequence.
- Inspect how one position can be restricted to a requested residue name.
- Compare total-score and block-pair interaction changes across a few variants.
- Separate an energy proxy from a physical folding or binding $\Delta\Delta G$.

> **GPU optional.** The small example runs on CPU; batching and larger design spaces benefit from CUDA.

## Setup

The setup uses the exact sampler construction exercised by TMol's packer tests and existing score/pack/minimize tutorial. The ten-residue slice keeps CPU execution practical.

In [ ]:
try:
    import google.colab  # noqa: F401
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True

if IN_COLAB:
    from urllib.request import urlopen

    exec(
        urlopen(
            "https://raw.githubusercontent.com/uw-ipd/tmol/"
            "kdidi/sphinx-docs/docs/tutorial/colab_setup.py"
        ).read(),
        globals(),
    )
    setup_colab(["tmol/tests/data/cif/1UBQ.cif"])

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from biotite.structure.io import load_structure

import tmol
from tmol import beta2016_score_function
from tmol.database import ParameterDatabase
from tmol.io.pose_stack_from_biotite import pose_stack_from_biotite
from tmol.pack.pack_rotamers import pack_rotamers
from tmol.pack.packer_task import PackerPalette, PackerTask
from tmol.pack.rotamer.dunbrack.dunbrack_chi_sampler import (
    create_dunbrack_sampler_from_database,
)
from tmol.pack.rotamer.fixed_aa_chi_sampler import FixedAAChiSampler
from tmol.pack.rotamer.include_current_sampler import IncludeCurrentSampler

SEED = 20260807
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
repo_root = Path.cwd()
if not (repo_root / "tmol/tests/data/cif/1UBQ.cif").exists():
    repo_root = Path(tmol.__file__).resolve().parents[1]
cif_path = repo_root / "tmol/tests/data/cif/1UBQ.cif"

param_db = ParameterDatabase.get_default()
atom_array = load_structure(str(cif_path), model=1, include_bonds=True)
protein_slice = atom_array[(atom_array.chain_id == "A") & (atom_array.res_id <= 10)]
pose_stack = pose_stack_from_biotite(
    protein_slice, device, param_db=param_db, no_optH=True
)
score_function = beta2016_score_function(device, param_db=param_db)


def show_table(frame):
    try:
        from itables import show
    except ImportError:
        display(frame)
        return None
    return show(frame)


def total_score(pose):
    scorer = score_function.render_whole_pose_scoring_module(pose)
    return float(scorer(pose.coords).detach().cpu()[0])

print(f"device={device}; input={cif_path.name}; slice blocks={pose_stack.max_n_blocks}")

## Fixed-sequence regional repacking

`PackerPalette` defines the residue-type space considered from each starting block. `PackerTask.restrict_to_repacking()` keeps each position's `name3` fixed. The task then uses Dunbrack rotamers, fixed-amino-acid chi samples, and the current conformation.

Here residues 3–8 (zero-based block indices 2–7) are allowed to repack. `disable_packing_by_block_mask()` receives `True` where packing should be disabled.

In [ ]:
palette = PackerPalette()
repack_task = PackerTask(pose_stack, palette)
repack_task.restrict_to_repacking()

packing_region = torch.zeros(
    (pose_stack.n_poses, pose_stack.max_n_blocks),
    dtype=torch.bool,
    device=device,
)
packing_region[:, 2:8] = True
repack_task.disable_packing_by_block_mask(~packing_region)
repack_task.add_conformer_sampler(
    create_dunbrack_sampler_from_database(param_db, device)
)
repack_task.add_conformer_sampler(FixedAAChiSampler())
repack_task.add_conformer_sampler(IncludeCurrentSampler())

start_score = total_score(pose_stack)
torch.manual_seed(SEED)
repacked_pose = pack_rotamers(pose_stack, score_function, repack_task)
repacked_score = total_score(repacked_pose)

pd.DataFrame(
    {
        "stage": ["input", "regional repack"],
        "beta2016": [start_score, repacked_score],
    }
)

**Expected observations.** The amino-acid sequence is unchanged, only blocks 3–8 are allowed to choose side-chain conformers, and the score often decreases after repacking. Simulated annealing is stochastic, so fixed seeds make the tutorial reproducible without promising one exact energy.

The viewer is interactive when the parent documentation environment provides `py3Dmol`.

In [ ]:
try:
    display(
        tmol.switchable_view(
            {"input": pose_stack, "regional repack": repacked_pose},
            notes={
                "input": f"weighted score: {start_score:.3f}",
                "regional repack": f"weighted score: {repacked_score:.3f}",
            },
            width=720,
            height=420,
        )
    )
except ImportError as exc:
    print("Interactive repacking comparison unavailable:", exc)

## Expand side-chain sampling

TMol's `PackerTask.or_expand_chi()` requests additional samples at approximately ±1 standard deviation around supported chi wells. Extra sampling enlarges the search rather than guaranteeing a lower score: the stochastic annealer still has finite effort, and `IncludeCurrentSampler` keeps the input conformation available.

In [ ]:
expanded_task = PackerTask(pose_stack, palette)
expanded_task.restrict_to_repacking()
expanded_task.disable_packing_by_block_mask(~packing_region)
expanded_task.add_conformer_sampler(
    create_dunbrack_sampler_from_database(param_db, device)
)
expanded_task.add_conformer_sampler(FixedAAChiSampler())
expanded_task.add_conformer_sampler(IncludeCurrentSampler())
expanded_task.or_expand_chi(1)
expanded_task.or_expand_chi(2)

torch.manual_seed(SEED)
expanded_repacked_pose = pack_rotamers(
    pose_stack, score_function, expanded_task
)
expanded_score = total_score(expanded_repacked_pose)
sampling_frame = pd.DataFrame(
    [
        {"sampling": "standard", "weighted_score": repacked_score},
        {"sampling": "expanded chi1 + chi2", "weighted_score": expanded_score},
    ]
)
show_table(sampling_frame)
display(
    tmol.switchable_view(
        {
            "standard rotamers": repacked_pose,
            "expanded chi1 + chi2": expanded_repacked_pose,
        },
        notes={
            "standard rotamers": f"score {repacked_score:.3f}",
            "expanded chi1 + chi2": f"score {expanded_score:.3f}",
        },
    )
)

## Deliberately small mutation scan

> **Inspected helper, not a built-in protocol.** `PackerTask` currently has no public method that restricts one position to one amino-acid identity. The helper below only turns `per_block_is_block_type_allowed` entries from `True` to `False`, preserving the task's monotonic-mask contract. Its field use was inspected against `tmol/pack/packer_task.py` and packer tests.

Every non-target position is restricted to its original block type. At the target, only considered block types with the requested `name3` remain. This is intentionally one position and four identities, not an exhaustive scan.

In [ ]:
def restrict_to_single_substitution(task, pose_index, block_index, name3):
    """Restrict task masks using inspected PackerTask fields; no entries are enabled."""
    considered = task.per_block_considered_block_types
    keep = task.per_block_considered_block_types_is_orig.detach().clone()
    target_considered = considered[pose_index, block_index].detach().cpu().tolist()
    target_keep = torch.tensor(
        [
            index >= 0 and task.pbt.active_block_types[index].name3 == name3
            for index in target_considered
        ],
        dtype=torch.bool,
        device=task.device,
    )
    if not torch.any(target_keep):
        raise ValueError(f"{name3!r} is not considered at block {block_index}")
    keep[pose_index, block_index] = target_keep
    task.per_block_is_block_type_allowed = torch.logical_and(
        task.per_block_is_block_type_allowed, keep
    )
    return task


def mutation_task(pose, block_index, name3):
    task = PackerTask(pose, PackerPalette())
    restrict_to_single_substitution(task, 0, block_index, name3)
    task.add_conformer_sampler(
        create_dunbrack_sampler_from_database(param_db, device)
    )
    task.add_conformer_sampler(FixedAAChiSampler())
    task.add_conformer_sampler(IncludeCurrentSampler())
    return task

In [ ]:
target_block = 4
original_type_index = int(pose_stack.block_type_ind64[0, target_block].item())
original_name3 = pose_stack.packed_block_types.active_block_types[
    original_type_index
].name3
variants = list(dict.fromkeys([original_name3, "ALA", "LEU", "PHE"]))


def target_interaction_proxy(pose, block_index):
    scorer = score_function.render_block_pair_scoring_module(pose)
    matrix = scorer(pose.coords, sum_terms=True, apply_weights=True)[0]
    partners = torch.ones(matrix.shape[0], dtype=torch.bool, device=matrix.device)
    partners[block_index] = False
    return float(
        (matrix[block_index, partners].sum() + matrix[partners, block_index].sum())
        .detach()
        .cpu()
    )

scan_rows = []
variant_poses = {}
for variant in variants:
    torch.manual_seed(SEED)
    task = mutation_task(pose_stack, target_block, variant)
    packed_variant = pack_rotamers(pose_stack, score_function, task)
    variant_poses[variant] = packed_variant
    scan_rows.append(
        {
            "block_index": target_block,
            "variant": variant,
            "total_score": total_score(packed_variant),
            "target_interaction_proxy": target_interaction_proxy(
                packed_variant, target_block
            ),
        }
    )

scan_frame = pd.DataFrame(scan_rows)
reference = scan_frame.loc[scan_frame["variant"] == original_name3].iloc[0]
scan_frame["delta_total_vs_repacked_wt"] = (
    scan_frame["total_score"] - reference["total_score"]
)
scan_frame["delta_proxy_vs_repacked_wt"] = (
    scan_frame["target_interaction_proxy"]
    - reference["target_interaction_proxy"]
)
show_table(scan_frame.sort_values("delta_total_vs_repacked_wt"))

In [ ]:
heatmap = scan_frame.set_index("variant")[["delta_proxy_vs_repacked_wt"]].T
fig, ax = plt.subplots(figsize=(7, 2.4))
image = ax.imshow(heatmap.to_numpy(), cmap="coolwarm", aspect="auto")
ax.set_xticks(range(len(heatmap.columns)), heatmap.columns)
ax.set_yticks([0], ["interaction proxy Δ"])
ax.set_title(f"Small scan at block {target_block} (WT {original_name3})")
fig.colorbar(image, ax=ax, label="beta2016 energy units")
plt.tight_layout()
plt.show()

**Expected observations.** Variants differ in both total score and their target-to-rest block-pair interaction. The lowest number in this tiny, stochastic exercise is not a prediction of stability.

> **Energy-proxy warning.** `delta_proxy_vs_repacked_wt` is the change in weighted block-pair score assigned between the target block and all other blocks after independent repacking. It omits ensemble averaging, unfolded/reference states, adequate structural relaxation, and solvent/experimental calibration. It is neither a physical $\Delta\Delta G$ nor TMol's built-in mutation protocol.

In [ ]:
best_variant = scan_frame.sort_values("total_score").iloc[0]["variant"]
variant_scores = scan_frame.set_index("variant")["total_score"].to_dict()
print("lowest total score in this tiny scan:", best_variant)
try:
    display(
        tmol.switchable_view(
            variant_poses,
            notes={
                variant: f"weighted score: {variant_scores[variant]:.3f}"
                for variant in variant_poses
            },
            width=720,
            height=420,
        )
    )
except ImportError as exc:
    print("Interactive mutation comparison unavailable:", exc)

## Rosetta comparison

Both systems separate a task describing allowed identities/conformers from the packing algorithm. Rosetta has mature `TaskOperation`, resfile, mover, and mutation-scan protocol layers. TMol exposes lower-level `PackerPalette`/`PackerTask` masks and GPU-oriented packer machinery, but no native point-mutation-scan protocol or Rosetta `ResidueSelector` equivalent.

For the complete Rosetta workflows, see [Optimizing Sidechains: The Packer](https://docs.rosettacommons.org/demos/latest/tutorials/Optimizing_Sidechains_The_Packer/Optimizing_Sidechains_The_Packer), [PyRosetta packing and regional relax](https://github.com/RosettaCommons/PyRosetta.notebooks/blob/master/notebooks/06.02-Packing-design-and-regional-relax.ipynb), [06.08 Point Mutation Scan](https://github.com/RosettaCommons/PyRosetta.notebooks/blob/master/notebooks/06.08-Point-Mutation-Scan.ipynb), and [16.01 distributed ddG/PSSM](https://github.com/RosettaCommons/PyRosetta.notebooks/blob/master/notebooks/16.01-PyData-ddG-pssm.ipynb).

## Limitations

- The per-position restriction helper uses inspected task fields rather than a stable high-level mutation API; revalidate it when packer internals change.
- The scan runs variants independently and does not batch them with a supported sequence-construction API.
- Only one position and a few identities are sampled, with bounded repacking and no backbone relaxation.
- No unfolded-state reference, ensemble, or calibrated thermodynamic model is included.
- Absolute ranking can depend on random seed, sampler coverage, score-function choice, and structural context.
- TMol has no built-in mutation-scan protocol; production workflows must own task construction, scheduling, provenance, and validation.

## Exercises

1. Change the regional repacking mask and verify that disabled blocks retain their input coordinates.
2. Inspect which `name3` values the default palette considers at the target before choosing variants.
3. Repeat the tiny scan for two additional seeds and report mean and range without calling the result a $\Delta\Delta G$.
4. Add Cartesian minimization after each packed variant and state how that changes the computational experiment.
5. Sketch an external scheduler that shards positions across GPUs while preserving one result row per position, identity, seed, and score-function version.

## References

- [Rosetta Packer tutorial](https://docs.rosettacommons.org/demos/latest/tutorials/Optimizing_Sidechains_The_Packer/Optimizing_Sidechains_The_Packer)
- [PyRosetta 06.01 Side Chain Conformations and Dunbrack Energies](https://github.com/RosettaCommons/PyRosetta.notebooks/blob/master/notebooks/06.01-Side-Chain-Conformations-and-Dunbrack-Energies.ipynb)
- [PyRosetta 06.02 Packing and regional relax](https://github.com/RosettaCommons/PyRosetta.notebooks/blob/master/notebooks/06.02-Packing-design-and-regional-relax.ipynb)
- [PyRosetta 06.08 Point Mutation Scan](https://github.com/RosettaCommons/PyRosetta.notebooks/blob/master/notebooks/06.08-Point-Mutation-Scan.ipynb)
- [PyRosetta 16.01 distributed ddG/PSSM](https://github.com/RosettaCommons/PyRosetta.notebooks/blob/master/notebooks/16.01-PyData-ddG-pssm.ipynb)
- [TMol repository](https://github.com/uw-ipd/tmol)